In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [ ]:

validation_df = pd.read_csv(
    "../data/processed/validation_data.csv",
    parse_dates=["snapshot_date"]
)

selected_model = joblib.load(
    "../models/logistic_regression_pipeline.pkl"
)

with open("../models/model_metadata.json", "r") as file:
    model_metadata = json.load(file)

print("Validation dataset shape:", validation_df.shape)
print("Selected model:", model_metadata["selected_model"])

print("\nValidation period:")
print(
    validation_df["snapshot_date"].min(),
    "to",
    validation_df["snapshot_date"].max()
)

print("\nModel loaded successfully.")

In [ ]:
feature_columns = (
    model_metadata["numerical_features"]
    + model_metadata["categorical_features"]
)

missing_columns = set(feature_columns + ["churn"]) - set(validation_df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

X_validation = validation_df[feature_columns].copy()
y_validation = validation_df["churn"].copy()

print("Validation features shape:", X_validation.shape)
print("Validation target shape:", y_validation.shape)

print("\nFeature count:", len(feature_columns))
print("\nTarget distribution:")
print(y_validation.value_counts())

In [ ]:
preprocessor = selected_model.named_steps["preprocessor"]

X_transformed = preprocessor.transform(X_validation)

log_features = preprocessor.transformers_[0][2]
numeric_features = preprocessor.transformers_[1][2]

categorical_encoder = (
    preprocessor.named_transformers_["categorical"]
    .named_steps["encoder"]
)

categorical_features = categorical_encoder.get_feature_names_out(
    preprocessor.transformers_[2][2]
)

transformed_feature_names = (
    list(log_features)
    + list(numeric_features)
    + list(categorical_features)
)

print("Original feature count:", X_validation.shape[1])
print("Transformed feature shape:", X_transformed.shape)
print("Transformed feature count:", len(transformed_feature_names))

In [ ]:
if X_transformed.shape[1] != len(transformed_feature_names):
    raise ValueError("Transformed feature names do not match transformed data.")

In [ ]:
X_background = preprocessor.transform(
    X_validation.sample(100, random_state=42)
)

X_sample = preprocessor.transform(
    X_validation.sample(1000, random_state=42)
)

print("Background sample shape:", X_background.shape)
print("SHAP sample shape:", X_sample.shape)

In [ ]:
explainer = shap.LinearExplainer(
    selected_model.named_steps["classifier"],
    X_background
)

shap_values = explainer(X_sample)

shap_values.feature_names = transformed_feature_names

print("SHAP values calculated successfully.")
print("SHAP values shape:", shap_values.values.shape)

In [ ]:
shap_importance = pd.DataFrame({
    "Feature": transformed_feature_names,
    "Mean Absolute SHAP": np.abs(shap_values.values).mean(axis=0)
}).sort_values(
    "Mean Absolute SHAP",
    ascending=False
)

print("Top 15 Churn Drivers:")
print(shap_importance.head(15).to_string(index=False))

In [ ]:
customer_index = 0

shap.plots.waterfall(
    shap_values[customer_index],
    max_display=15
)

In [ ]:
shap.plots.bar(
    shap_values,
    max_display=15
)

In [ ]:
shap.plots.beeswarm(
    shap_values,
    max_display=15
)